# B1 baseline on Colab GPU

Before running: in Colab choose **Runtime → Change runtime type → T4 GPU**.

Needs two files in your Google Drive, in a folder called `mf`:
- `processed.zip` (the processed double-delta data)
- `mfb.zip` (the code folder)

Results are written back to `MyDrive/mf/baselines.jsonl`, so nothing is lost if Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# copy and unpack data + code (about 1-2 minutes)
!cp /content/drive/MyDrive/mf/processed.zip /content/drive/MyDrive/mf/mfb.zip /content/
!cd /content && unzip -q -o processed.zip && unzip -q -o mfb.zip
!ls /content/processed | head; ls /content/processed/snap | wc -l

In [ ]:
# PyTorch is already installed on Colab; add PyTorch Geometric
!pip -q install torch_geometric
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch runtime to T4 GPU')

In [ ]:
# quick test first: 2 epochs, should finish in a minute or two
%cd /content
!python -m mfb.train --data /content/processed --mode all --epochs 2 --out /content/drive/MyDrive/mf/test_run.jsonl

In [ ]:
# B1: all 256 training wings, 50 epochs (then try 200 if time allows)
%cd /content
!python -m mfb.train --data /content/processed --mode all --epochs 50 --out /content/drive/MyDrive/mf/baselines.jsonl

In [ ]:
# show results
!cat /content/drive/MyDrive/mf/baselines.jsonl

## Paper-matched setting: 5 angles (11, 13, 15, 17, 19 deg)
Training sizes become 40-1280 cases and the test set 80 cases, as in Shen & Alonso Sec. 4.

In [ ]:
%cd /content
P='/content/drive/MyDrive/mf/paper_setting.jsonl'
!python -m mfb.train --data /content/processed --b0 --aoas 11 13 15 17 19 --out $P

In [ ]:
# B3: error vs training size, Sobol order (about 10 minutes on T4)
!python -m mfb.train --data /content/processed --mode sobol --n-geoms 8 16 32 64 128 256 --epochs 50 --aoas 11 13 15 17 19 --out $P

In [ ]:
import pandas as pd
d = pd.read_json(P, lines=True); print(d[[c for c in ['baseline','n_snapshots','n_test','mse_cp','cl_abs_err','cm_abs_err','train_seconds'] if c in d]].to_string())